# FlashAttention GPU Wrapper

This notebook wraps the current `cs336_systems/implementations/flash_attention.py` implementation so you can test it on a GPU runtime. It follows the same Colab-style flow as `attention_compile_benchmarks.ipynb`:

1. clone or refresh the repo
2. install dependencies needed by `flash_attention.py`
3. import the current FlashAttention implementations
4. run direct forward comparisons against a PyTorch reference
5. run the focused `pytest` checks for the assignment tests


In [2]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

REPO_URL = os.environ.get("ASSIGNMENT2_REPO_URL", "https://github.com/Eden-kk/assignment2-systems.git")
REPO_ROOT = Path("/content/assignment2-systems")

if REPO_ROOT.exists():
    print(f"Refreshing existing repo at {REPO_ROOT}")
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_ROOT, check=True, timeout=300)
    subprocess.run(["git", "checkout", "main"], cwd=REPO_ROOT, check=True, timeout=300)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_ROOT, check=True, timeout=300)
else:
    print(f"Cloning {REPO_URL} into {REPO_ROOT} ...")
    subprocess.run(["git", "clone", "--depth", "1", "--progress", REPO_URL, str(REPO_ROOT)], check=True, timeout=300)

os.chdir(REPO_ROOT)
head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
print(f"Repo root: {REPO_ROOT}")
print(f"Checked out commit: {head}")


Cloning https://github.com/Eden-kk/assignment2-systems.git into /content/assignment2-systems ...
Repo root: /content/assignment2-systems
Checked out commit: 2be7db5


In [3]:
%cd /content/assignment2-systems
%pip uninstall -y -q torchaudio torchvision torchtext fastai timm
%pip install -q einops triton
%pip install -q -e ./cs336-basics -e .


/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 3.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 5.0 MB/s eta 0:00:

In [1]:
import os
import shutil

import psutil
import torch

try:
    import triton
    triton_version = triton.__version__
except Exception as exc:
    triton_version = f"unavailable: {exc}"

def cuda_runtime_usable() -> tuple[bool, str]:
    if not torch.cuda.is_available():
        return False, "CUDA is not available in this runtime."
    try:
        _ = torch.tensor([1], device="cuda") + 1
        torch.cuda.synchronize()
        name = torch.cuda.get_device_name(0)
        capability = torch.cuda.get_device_capability(0)
        return True, f"Using CUDA on {name} with capability {capability}."
    except Exception as exc:
        return False, f"CUDA is present but unusable with this Torch build: {exc}"

cuda_ok, cuda_message = cuda_runtime_usable()
print("Torch version:", torch.__version__)
print("Triton version:", triton_version)
print("CUDA available:", torch.cuda.is_available())
print("CUDA usable:", cuda_ok)
print("CUDA status:", cuda_message)
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))
vm = psutil.virtual_memory()
disk = shutil.disk_usage("/")
print("RAM total (GB):", round(vm.total / 1e9, 2))
print("Disk free (GB):", round(disk.free / 1e9, 2))
if not cuda_ok:
    raise RuntimeError("This notebook expects a CUDA GPU runtime.")


Torch version: 2.10.0+cu128
Triton version: 3.6.0
CUDA available: True
CUDA usable: True
CUDA status: Using CUDA on NVIDIA H100 80GB HBM3 with capability (9, 0).
COLAB_RELEASE_TAG: release-colab-external-images_20260326-060101_RC00
COLAB_GPU: 1
GPU: NVIDIA H100 80GB HBM3
GPU capability: (9, 0)
RAM total (GB): 247.01
Disk free (GB): 206.25


In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import torch
from einops import einsum

repo_root = Path("/content/assignment2-systems")
basics_root = repo_root / "cs336-basics"
systems_root = repo_root / "cs336_systems"
implementations_root = systems_root / "implementations"

for path in (repo_root, basics_root, systems_root, implementations_root):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

importlib.invalidate_caches()

import cs336_systems.implementations.flash_attention as flash_attention
import tests.adapters as adapters

FlashAttention2PyTorch = flash_attention.FlashAttention2PyTorch
FlashAttention2Triton = flash_attention.FlashAttention2Triton

def attention_and_lse_reference(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, is_causal: bool = False):
    n_queries = q.shape[-2]
    n_keys = k.shape[-2]
    d = q.shape[-1]
    scale = 1 / (d ** 0.5)
    scores = einsum(q, k, "... q d, ... k d -> ... q k") * scale
    if is_causal:
        mask = torch.arange(n_queries, device=scores.device)[None, :, None] >= torch.arange(n_keys, device=scores.device)[None, None, :]
        scores = torch.where(mask, scores, torch.full_like(scores, -1e6))
    probs = torch.softmax(scores, dim=-1)
    output = einsum(probs, v, "... q k, ... k d -> ... q d")
    lse = torch.logsumexp(scores, dim=-1)
    return output, lse

def make_attn_inputs(device: str = "cuda", dtype: torch.dtype = torch.float32):
    torch.random.manual_seed(0)
    batch_size = 4
    n_queries = 128
    n_keys = 128
    d_model = 64
    q = torch.randn(batch_size, n_queries, d_model, device=device, dtype=dtype, requires_grad=True)
    k = torch.randn(batch_size, n_keys, d_model, device=device, dtype=dtype, requires_grad=True)
    v = torch.randn(batch_size, n_keys, d_model, device=device, dtype=dtype, requires_grad=True)
    return q, k, v

def run_forward_check(impl_cls, *, device: str = "cuda", dtype: torch.dtype = torch.float32, is_causal: bool = False) -> dict[str, object]:
    q, k, v = make_attn_inputs(device=device, dtype=dtype)
    out = impl_cls.apply(q, k, v, is_causal)
    saved = list(out.grad_fn.saved_tensors)
    l_candidates = [tensor for tensor in saved if tuple(tensor.shape) == (q.shape[0], q.shape[1])]
    if len(l_candidates) != 1:
        raise RuntimeError(f"Expected exactly one saved L tensor, found {len(l_candidates)}")
    l = l_candidates[0]
    out_ref, l_ref = attention_and_lse_reference(q, k, v, is_causal=is_causal)
    return {
        "impl": impl_cls.__name__,
        "device": device,
        "dtype": str(dtype),
        "is_causal": is_causal,
        "max_abs_diff_o": float((out - out_ref).abs().max().item()),
        "max_abs_diff_l": float((l - l_ref).abs().max().item()),
    }

print("Loaded FlashAttention module from:", flash_attention.__file__)


In [ ]:
DEVICE = "cuda"
DTYPE = torch.float32
CAUSAL_OPTIONS = [False, True]

print("Device:", DEVICE)
print("Dtype:", DTYPE)
print("Causal options:", CAUSAL_OPTIONS)


## Direct Forward Checks

These cells call the current autograd.Function implementations directly and compare their output and saved `L` tensor against a reference implementation.


In [ ]:
rows = []

for impl_cls in [FlashAttention2PyTorch, FlashAttention2Triton]:
    for is_causal in CAUSAL_OPTIONS:
        try:
            rows.append(run_forward_check(impl_cls, device=DEVICE, dtype=DTYPE, is_causal=is_causal))
        except Exception as exc:
            rows.append({
                "impl": impl_cls.__name__,
                "device": DEVICE,
                "dtype": str(DTYPE),
                "is_causal": is_causal,
                "max_abs_diff_o": None,
                "max_abs_diff_l": None,
                "status": f"failed: {exc}",
            })

direct_df = pd.DataFrame(rows)
if "status" not in direct_df.columns:
    direct_df["status"] = "ok"
direct_df


## Focused Pytest Checks

These cells run the assignment's targeted forward tests using the repo's `uv` environment.


In [ ]:
!uv run pytest -k test_flash_forward_pass_pytorch -q


In [ ]:
!uv run pytest -k test_flash_forward_pass_triton -q
